In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

In [5]:
# Load the cleaned feature files from Day 2
batsman_stats = pd.read_csv(r"C:\Users\dhruv\playXI\data\batsman_stats.csv")
bowler_stats = pd.read_csv(r"C:\Users\dhruv\playXI\data\bowler_stats.csv")

print("Batsman stats shape:", batsman_stats.shape)
print("Bowler stats shape:", bowler_stats.shape)
print("Data loaded successfully!")


Batsman stats shape: (17633, 5)
Bowler stats shape: (13845, 6)
Data loaded successfully!


In [6]:
# Step 1: Calculate fantasy points for batsmen
def calculate_batting_points(row):
    points = 0
    
    # 1 point per run
    points += row['total_runs']
    
    # Bonus for half century
    if row['total_runs'] >= 50:
        points += 8
    
    # Bonus for century
    if row['total_runs'] >= 100:
        points += 16
    
    # Bonus for strike rate (if faced more than 10 balls)
    if row['balls_faced'] >= 10:
        if row['strike_rate'] >= 170:
            points += 6
        elif row['strike_rate'] >= 150:
            points += 4
        elif row['strike_rate'] >= 130:
            points += 2
    
    return points

# Apply this function to every row in batsman_stats
batsman_stats['fantasy_points'] = batsman_stats.apply(calculate_batting_points, axis=1)

print("Fantasy points added!")
print(batsman_stats.head(10))


Fantasy points added!
   matchId          batsman  total_runs  balls_faced  strike_rate  \
0   335982        AA Noffke           9           10    90.000000   
1   335982          B Akhil           0            2     0.000000   
2   335982      BB McCullum         158           73   216.438356   
3   335982         CL White           6           10    60.000000   
4   335982        DJ Hussey          12           12   100.000000   
5   335982        JH Kallis           8            7   114.285714   
6   335982       MV Boucher           7            9    77.777778   
7   335982  Mohammad Hafeez           5            3   166.666667   
8   335982          P Kumar          18           15   120.000000   
9   335982         R Dravid           2            3    66.666667   

   fantasy_points  
0               9  
1               0  
2             188  
3               6  
4              12  
5               8  
6               7  
7               5  
8              18  
9               2 

In [7]:
# Step 2: Calculate fantasy points for bowlers
def calculate_bowling_points(row):
    points = 0
    
    # 25 points per wicket
    points += row['wickets'] * 25
    
    # Bonus for 3 wicket haul
    if row['wickets'] >= 3:
        points += 4
    
    # Bonus for 4 wicket haul
    if row['wickets'] >= 4:
        points += 8
    
    # Bonus for good economy (only if bowled 12+ balls)
    if row['balls_bowled'] >= 12:
        if row['economy'] <= 5:
            points += 6
        elif row['economy'] <= 6:
            points += 4
        elif row['economy'] <= 7:
            points += 2
    
    return points

# Apply to every row in bowler_stats
bowler_stats['fantasy_points'] = bowler_stats.apply(calculate_bowling_points, axis=1)

print("Bowler fantasy points added!")
print(bowler_stats.head(10))

Bowler fantasy points added!
   matchId      bowler  runs_conceded  balls_bowled    economy  wickets  \
0   335982   AA Noffke             35            24   8.750000      1.0   
1   335982  AB Agarkar             21            24   5.250000      3.0   
2   335982    AB Dinda              7            18   2.333333      2.0   
3   335982    CL White             22             6  22.000000      0.0   
4   335982    I Sharma              6            18   2.000000      1.0   
5   335982   JH Kallis             47            24  11.750000      1.0   
6   335982   LR Shukla              9             7   7.714286      1.0   
7   335982     P Kumar             37            24   9.250000      0.0   
8   335982    SB Joshi             26            18   8.666667      0.0   
9   335982  SC Ganguly             20            24   5.000000      2.0   

   fantasy_points  
0            25.0  
1            83.0  
2            56.0  
3             0.0  
4            31.0  
5            25.0  
6    

In [9]:
# Step 3: Add Recent Form — average fantasy points in last 5 matches
# First sort by matchId so matches are in order
batsman_stats = batsman_stats.sort_values('matchId')

# Calculate rolling average of last 5 matches for each batsman
batsman_stats['recent_form'] = (
    batsman_stats.groupby('batsman')['fantasy_points']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# Fill NaN recent form with overall average (for players with very few matches)
batsman_stats['recent_form'] = batsman_stats['recent_form'].fillna(
    batsman_stats['fantasy_points'].mean()
)

print("Recent form added for batsmen!")
print(batsman_stats.head(10))


Recent form added for batsmen!
   matchId          batsman  total_runs  balls_faced  strike_rate  \
0   335982        AA Noffke           9           10    90.000000   
1   335982          B Akhil           0            2     0.000000   
2   335982      BB McCullum         158           73   216.438356   
3   335982         CL White           6           10    60.000000   
4   335982        DJ Hussey          12           12   100.000000   
5   335982        JH Kallis           8            7   114.285714   
6   335982       MV Boucher           7            9    77.777778   
7   335982  Mohammad Hafeez           5            3   166.666667   
8   335982          P Kumar          18           15   120.000000   
9   335982         R Dravid           2            3    66.666667   

   fantasy_points  recent_form  
0               9    22.262406  
1               0    22.262406  
2             188    22.262406  
3               6    22.262406  
4              12    22.262406  
5          

In [10]:
# Step 4: Add Recent Form for bowlers
bowler_stats = bowler_stats.sort_values('matchId')

bowler_stats['recent_form'] = (
    bowler_stats.groupby('bowler')['fantasy_points']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

bowler_stats['recent_form'] = bowler_stats['recent_form'].fillna(
    bowler_stats['fantasy_points'].mean()
)

print("Recent form added for bowlers!")
print(bowler_stats.head(10))


Recent form added for bowlers!
   matchId      bowler  runs_conceded  balls_bowled    economy  wickets  \
0   335982   AA Noffke             35            24   8.750000      1.0   
1   335982  AB Agarkar             21            24   5.250000      3.0   
2   335982    AB Dinda              7            18   2.333333      2.0   
3   335982    CL White             22             6  22.000000      0.0   
4   335982    I Sharma              6            18   2.000000      1.0   
5   335982   JH Kallis             47            24  11.750000      1.0   
6   335982   LR Shukla              9             7   7.714286      1.0   
7   335982     P Kumar             37            24   9.250000      0.0   
8   335982    SB Joshi             26            18   8.666667      0.0   
9   335982  SC Ganguly             20            24   5.000000      2.0   

   fantasy_points  recent_form  
0            25.0    24.763236  
1            83.0    24.763236  
2            56.0    24.763236  
3          

In [11]:
# Step 5: Prepare data for ML model — Batsman model
# Features = inputs the model uses to predict
# Target = what we want to predict (fantasy points)

batsman_features = batsman_stats[['total_runs', 'balls_faced', 'strike_rate', 'recent_form']]
batsman_target = batsman_stats['fantasy_points']

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    batsman_features, batsman_target, test_size=0.2, random_state=42
)

print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)


Training data size: (14106, 4)
Testing data size: (3527, 4)


In [12]:
# Step 6: Train the Random Forest model for batsmen
batsman_model = RandomForestRegressor(n_estimators=100, random_state=42)
batsman_model.fit(X_train, y_train)

# Test the model
y_pred = batsman_model.predict(X_test)

# Check accuracy
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Batsman Model Results:")
print(f"Mean Absolute Error: {mae:.2f} points")
print(f"R2 Score: {r2:.2f}")


Batsman Model Results:
Mean Absolute Error: 0.02 points
R2 Score: 1.00


In [13]:
# Step 7: Retrain with only features we know BEFORE the match
# We can only use recent_form as our main predictor
# because total_runs/strike_rate are things that happen DURING the match

batsman_features_real = batsman_stats[['recent_form']]
batsman_target_real = batsman_stats['fantasy_points']

# Split data
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    batsman_features_real, batsman_target_real, test_size=0.2, random_state=42
)

# Train model
batsman_model2 = RandomForestRegressor(n_estimators=100, random_state=42)
batsman_model2.fit(X_train2, y_train2)

# Test model
y_pred2 = batsman_model2.predict(X_test2)

# Check accuracy
mae2 = mean_absolute_error(y_test2, y_pred2)
r2_2 = r2_score(y_test2, y_pred2)

print("Batsman Model Results (Realistic):")
print(f"Mean Absolute Error: {mae2:.2f} points")
print(f"R2 Score: {r2_2:.2f}")


Batsman Model Results (Realistic):
Mean Absolute Error: 18.00 points
R2 Score: 0.05


In [14]:
# Step 8: Add more pre-match features
# Career average runs per match for each batsman
career_avg = batsman_stats.groupby('batsman')['total_runs'].mean().reset_index()
career_avg.columns = ['batsman', 'career_avg_runs']

# Career average strike rate for each batsman  
career_sr = batsman_stats.groupby('batsman')['strike_rate'].mean().reset_index()
career_sr.columns = ['batsman', 'career_avg_sr']

# Merge these back into batsman_stats
batsman_stats = batsman_stats.merge(career_avg, on='batsman')
batsman_stats = batsman_stats.merge(career_sr, on='batsman')

print("New features added!")
print(batsman_stats.columns.tolist())


New features added!
['matchId', 'batsman', 'total_runs', 'balls_faced', 'strike_rate', 'fantasy_points', 'recent_form', 'career_avg_runs', 'career_avg_sr']


In [15]:
# Step 9: Retrain with better features
batsman_features_v2 = batsman_stats[['recent_form', 'career_avg_runs', 'career_avg_sr']]
batsman_target_v2 = batsman_stats['fantasy_points']

# Split data
X_train3, X_test3, y_train3, y_test3 = train_test_split(
    batsman_features_v2, batsman_target_v2, test_size=0.2, random_state=42
)

# Train model
batsman_model3 = RandomForestRegressor(n_estimators=100, random_state=42)
batsman_model3.fit(X_train3, y_train3)

# Test model
y_pred3 = batsman_model3.predict(X_test3)

# Check accuracy
mae3 = mean_absolute_error(y_test3, y_pred3)
r2_3 = r2_score(y_test3, y_pred3)

print("Batsman Model v2 Results:")
print(f"Mean Absolute Error: {mae3:.2f} points")
print(f"R2 Score: {r2_3:.2f}")
print("\nImprovement from v1:")
print(f"MAE improved by: {mae2 - mae3:.2f} points")

Batsman Model v2 Results:
Mean Absolute Error: 18.57 points
R2 Score: -0.07

Improvement from v1:
MAE improved by: -0.57 points


In [16]:
# Step 10: Train bowler model
bowler_features = bowler_stats[['recent_form']]
bowler_target = bowler_stats['fantasy_points']

# Split data
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    bowler_features, bowler_target, test_size=0.2, random_state=42
)

# Train model
bowler_model = RandomForestRegressor(n_estimators=100, random_state=42)
bowler_model.fit(X_train_b, y_train_b)

# Test model
y_pred_b = bowler_model.predict(X_test_b)

# Check accuracy
mae_b = mean_absolute_error(y_test_b, y_pred_b)
r2_b = r2_score(y_test_b, y_pred_b)

print("Bowler Model Results:")
print(f"Mean Absolute Error: {mae_b:.2f} points")
print(f"R2 Score: {r2_b:.2f}")

Bowler Model Results:
Mean Absolute Error: 20.97 points
R2 Score: -0.02


In [17]:
# Step 11: Save both models to disk
import pickle

# Save batsman model
with open(r"C:\Users\dhruv\playXI\src\batsman_model.pkl", 'wb') as f:
    pickle.dump(batsman_model2, f)

# Save bowler model
with open(r"C:\Users\dhruv\playXI\src\bowler_model.pkl", 'wb') as f:
    pickle.dump(bowler_model, f)

print("Both models saved successfully!")
print("batsman_model.pkl saved in src/")
print("bowler_model.pkl saved in src/")

Both models saved successfully!
batsman_model.pkl saved in src/
bowler_model.pkl saved in src/
